# 威廉·夏普风格分析 - 完整复现

本Notebook复现华泰证券2020年金工研报《威廉·夏普风格分析基于基金收益率，能实现对基金风格的高频跟踪》

## 核心算法

**优化目标：**
```
min Σ(R - Σβ_j * x_j)^2
s.t. Σβ_j = 1, 0 ≤ β_j ≤ 1
```

其中：
- R: 基金收益率序列
- x_j: 第j个风格指数收益率序列
- β_j: 基金在第j个风格指数上的暴露

**SDS风格漂移指标（Idzorek方法）：**
```
SDS = √[Var(β_1) + Var(β_2) + ... + Var(β_n)]
```

## 1. 环境设置与导入

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 导入自定义模块
sys.path.insert(0, os.path.join(os.getcwd(), 'source'))
from source.data_loader import StyleDataLoader
from source.factor import SharpeStyleModel, RollingStyleAnalyzer, compute_sds
from source.backtest import StyleDriftDetector, StyleBacktest
from source.plot import StyleVisualizer
from source.utils import (
    generate_style_report, save_results_to_json,
    calculate_performance_metrics, format_date, load_index_name_mapping
)

print("[OK] 环境初始化完成")

## 2. 参数配置

In [ ]:
# 基金配置
FUND_CODE = '021181'  # 中欧价值精选混合A
FUND_NAME = '中欧价值精选混合A'

# 时间范围
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=365)

START_DATE_STR = START_DATE.strftime('%Y%m%d')
END_DATE_STR = END_DATE.strftime('%Y%m%d')

# 风格指数配置（根据研报推荐）
STYLE_INDICES = [
    '000300.SH',   # 沪深300 - 大盘
    '000905.SH',   # 中证500 - 中盘
    '000918.SH',   # 沪深300成长
    '000919.SH',   # 沪深300价值
]

# 指数名称映射
index_names = load_index_name_mapping()

print(f"基金代码: {FUND_CODE}")
print(f"分析区间: {START_DATE_STR} - {END_DATE_STR}")
print(f"风格指数:")
for idx in STYLE_INDICES:
    print(f"  {idx}: {index_names.get(idx, idx)}")

## 3. 数据获取

In [ ]:
# 初始化数据加载器
loader = StyleDataLoader()

# 获取基金净值数据
try:
    fund_df = loader.get_fund_nav(FUND_CODE, START_DATE_STR, END_DATE_STR)
    fund_returns = fund_df.set_index('日期')['daily_return']
    print(f"[OK] 获取基金数据: {len(fund_df)} 条\n")
except Exception as e:
    print(f"[WARN] 获取真实数据失败: {e}")
    print("[INFO] 使用模拟数据...\n")
    fund_df, _ = loader.create_mock_data(FUND_CODE, periods=252, style_indices=STYLE_INDICES)
    fund_returns = fund_df.set_index('日期')['daily_return']

# 显示基金净值走势
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fund_df['日期'], fund_df['nav'], linewidth=1.5)
ax.set_xlabel('日期')
ax.set_ylabel('单位净值')
ax.set_title(f'{FUND_NAME} ({FUND_CODE}) 净值走势')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 获取风格指数数据
try:
    index_df = loader.get_index_returns(STYLE_INDICES, START_DATE_STR, END_DATE_STR)
    print(f"[OK] 获取指数数据: {len(index_df)} 个交易日\n")
except Exception as e:
    print(f"[WARN] 获取指数数据失败: {e}")
    print("[INFO] 使用模拟数据...\n")
    _, index_df = loader.create_mock_data(FUND_CODE, periods=len(fund_returns), style_indices=STYLE_INDICES)

# 对齐日期
common_dates = fund_returns.index.intersection(index_df.index)
fund_returns = fund_returns.loc[common_dates]
index_df = index_df.loc[common_dates]

print(f"对齐后数据: {len(common_dates)} 个交易日")

## 4. 威廉·夏普风格分析

In [ ]:
# 创建风格分析模型
model = SharpeStyleModel(STYLE_INDICES)

# 拟合模型
style_result = model.fit(fund_returns, index_df)

print("=" * 60)
print("风格分析结果")
print("=" * 60)
print("\n风格暴露系数:")
for idx, exp in style_result['exposures'].sort_values(ascending=False).items():
    bar = "█" * int(exp * 50)
    print(f"  {index_names.get(idx, idx):15s}: {exp:.4f} ({exp:.2%}) {bar}")

print(f"\n模型拟合优度:")
print(f"  R² = {style_result['r_squared']:.4f}")
print(f"  跟踪误差 = {style_result['tracking_error']:.4f} (年化)")

style_label = model.get_style_label()
print(f"\n风格标签: {style_label}")

## 5. 可视化分析

In [ ]:
# 初始化可视化工具
visualizer = StyleVisualizer()

# 5.1 风格暴露条形图
fig1 = visualizer.plot_style_exposure(
    style_result['exposures'],
    title=f"{FUND_NAME} ({FUND_CODE}) 风格暴露分析"
)
plt.show()

In [ ]:
# 5.2 实际收益 vs 拟合收益对比
fig2 = visualizer.plot_return_comparison(
    fund_returns,
    style_result['fitted_returns'],
    title=f"{FUND_NAME} ({FUND_CODE}) 实际收益 vs 风格拟合收益"
)
plt.show()

## 6. 风格漂移检测 (SDS指标)

In [ ]:
# 创建风格漂移检测器
detector = StyleDriftDetector()

# 分析子区间
sub_period_df = detector.analyze_sub_periods(
    fund_returns, index_df, STYLE_INDICES, n_periods=4
)

print("=" * 60)
print("各子区间风格分析")
print("=" * 60)
display(sub_period_df)

# 计算SDS指标
sds_score = detector.compute_sds()
print(f"\nSDS风格漂移指标: {sds_score:.4f}")

if sds_score < 0.1:
    print("  → 风格高度稳定 ✓")
elif sds_score < 0.2:
    print("  → 风格相对稳定")
elif sds_score < 0.3:
    print("  → 风格存在一定波动 ⚠")
else:
    print("  → 风格漂移风险较高 ⚠⚠")

In [ ]:
# 检查风格漂移
drift_result = detector.check_style_drift(sub_period_df)

print("\n风格漂移检测结果:")
print(f"  是否漂移: {drift_result['has_drift']}")
print(f"  一致性评分: {drift_result['consistency_score']:.4f}")
print(f"  分析: {drift_result['analysis']}")

In [ ]:
# 绘制SDS分析图
fig3 = visualizer.plot_sds_analysis(sub_period_df, sds_score)
plt.show()

## 7. 滚动窗口风格分析

In [ ]:
# 创建滚动分析器
rolling_analyzer = RollingStyleAnalyzer(window=63, step=21)  # 3个月窗口，1个月步长

# 执行滚动分析
rolling_results = rolling_analyzer.analyze(
    fund_returns, index_df, STYLE_INDICES
)

print(f"[OK] 完成 {len(rolling_results)} 个窗口的分析")
display(rolling_results.head(10))

In [ ]:
# 绘制滚动风格暴露变化
fig4 = visualizer.plot_style_timeline(
    rolling_results,
    STYLE_INDICES,
    title=f"{FUND_NAME} 风格暴露时序变化"
)
plt.show()

## 8. 绩效指标计算

In [ ]:
# 计算绩效指标
perf_metrics = calculate_performance_metrics(fund_returns)

print("=" * 60)
print("绩效指标")
print("=" * 60)
print(f"\n年化收益率: {perf_metrics['ann_return']:.2%}")
print(f"年化波动率: {perf_metrics['ann_volatility']:.2%}")
print(f"夏普比率: {perf_metrics['sharpe_ratio']:.4f}")
print(f"最大回撤: {perf_metrics['max_drawdown']:.2%}")
print(f"卡玛比率: {perf_metrics['calmar_ratio']:.4f}")
print(f"胜率: {perf_metrics['win_rate']:.2%}")
print(f"盈亏比: {perf_metrics['profit_loss_ratio']:.4f}")
print(f"总收益率: {perf_metrics['total_return']:.2%}")

## 9. 生成完整报告

In [ ]:
# 生成文本报告
report_text = generate_style_report(
    FUND_CODE, FUND_NAME,
    style_result, drift_result, perf_metrics
)

print(report_text)

In [ ]:
# 保存结果
import os
os.makedirs('output', exist_ok=True)

# 保存JSON
results = {
    'fund_code': FUND_CODE,
    'fund_name': FUND_NAME,
    'analysis_date': datetime.now().strftime('%Y-%m-%d'),
    'period': {'start': START_DATE_STR, 'end': END_DATE_STR},
    'style_analysis': {
        'exposures': style_result['exposures'].to_dict(),
        'r_squared': style_result['r_squared'],
        'tracking_error': style_result['tracking_error'],
        'style_label': style_label
    },
    'drift_analysis': drift_result,
    'performance': perf_metrics
}

save_results_to_json(results, f'output/{FUND_CODE}_results.json')

# 保存报告
with open(f'output/{FUND_CODE}_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)

print("[OK] 结果已保存到 output/ 目录")

## 10. 扩展：对比多只基金

In [ ]:
# 定义多只基金进行对比
funds_to_compare = [
    {'code': '021181', 'name': '中欧价值精选A', 'nominal_style': '价值型'},
    {'code': '008404', 'name': '华泰紫金泰盈混合', 'nominal_style': '均衡型'},
    # 添加更多基金...
]

comparison_results = []

for fund_info in funds_to_compare:
    try:
        # 获取数据
        fund_df = loader.get_fund_nav(fund_info['code'], START_DATE_STR, END_DATE_STR)
        fund_ret = fund_df.set_index('日期')['daily_return']
        
        # 对齐指数数据
        common_dates = fund_ret.index.intersection(index_df.index)
        fund_ret = fund_ret.loc[common_dates]
        idx_ret = index_df.loc[common_dates]
        
        # 风格分析
        m = SharpeStyleModel(STYLE_INDICES)
        res = m.fit(fund_ret, idx_ret)
        
        comparison_results.append({
            'fund_code': fund_info['code'],
            'fund_name': fund_info['name'],
            'nominal_style': fund_info['nominal_style'],
            'actual_style': m.get_style_label(),
            'r_squared': res['r_squared'],
            'tracking_error': res['tracking_error'],
            **res['exposures'].to_dict()
        })
    except Exception as e:
        print(f"[WARN] {fund_info['code']} 分析失败: {e}")

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

## 总结

本Notebook完整复现了威廉·夏普风格分析模型：

1. **核心算法**：通过带约束的二次规划求解风格暴露系数
2. **SDS指标**：基于Idzorek方法量化风格漂移程度
3. **可视化**：风格暴露、收益对比、时序变化等多维度图表
4. **应用**：可用于基金风格识别、风格漂移监控、FOF组合构建

**注意事项**：
- 风格指数的选择会影响分析结果
- R²较低时说明基金风格不明显或模型解释力不足
- SDS指标需要足够长的历史数据才有意义